# Satellite Image Dehazing — Demo Notebook

This notebook walks through the full dehazing pipeline step by step,
visualising each intermediate stage and explaining the physics.

**Pipeline stages:**
- Stage 0 — Sky mask detection
- Stage 1 — Dark Channel Prior (DCP)
- Stage 2 — Guided filter refinement
- Stage 3 — Scene recovery
- Stage 4 — Mild CLAHE polish
- Stage 5 — Contrast stretch + sharpening

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join('..', 'src'))

import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from enhance import PipelineConfig, run_pipeline
from enhance.utils import show_comparison, show_pipeline, compute_psnr

## 1. Load a hazy satellite image

Set `IMAGE_PATH` to any hazy satellite image.  See `../samples/README.md`
for recommended datasets and quick-test sources.

In [ ]:
IMAGE_PATH = "../samples/your_hazy_image.jpg"  # <-- change this

image = cv2.imread(IMAGE_PATH)
if image is None:
    raise FileNotFoundError(f"Could not load image: {IMAGE_PATH}")

print(f"Image shape: {image.shape}")
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title("Input (Hazy)")
plt.axis('off')
plt.show()

## 2. Run the full pipeline with defaults

In [ ]:
config = PipelineConfig()  # all default values
result, stages = run_pipeline(image, config, verbose=True)

## 3. Before / after comparison

In [ ]:
show_comparison(image, result, title="Satellite Dehazing Result")

## 4. Visualise all pipeline stages

In [ ]:
show_pipeline(stages)

## 5. Inspect the transmission map

The transmission map `t(x)` represents the fraction of scene light
that reaches the camera.  Low values (dark areas in the map) indicate
dense haze; high values (bright) indicate clear regions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(stages["1 · Raw Transmission"], cmap='viridis', vmin=0, vmax=1)
axes[0].set_title("Raw Transmission t(x)")
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(stages["2 · Refined Transmission"], cmap='viridis', vmin=0, vmax=1)
axes[1].set_title("Refined Transmission t(x) — after guided filter")
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()

## 6. Parameter tuning

Experiment with key parameters to see how they affect the result.

In [ ]:
configs = {
    "omega=0.95 (default)": PipelineConfig(dcp_omega=0.95),
    "omega=0.75 (less removal)": PipelineConfig(dcp_omega=0.75),
    "omega=1.00 (full removal)": PipelineConfig(dcp_omega=1.00),
}

fig, axes = plt.subplots(1, len(configs), figsize=(7 * len(configs), 6))
for ax, (label, cfg) in zip(axes, configs.items()):
    r, _ = run_pipeline(image, cfg)
    ax.imshow(cv2.cvtColor(r, cv2.COLOR_BGR2RGB))
    ax.set_title(label)
    ax.axis('off')

plt.suptitle("Effect of omega (haze retention factor)", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Evaluate with PSNR (if ground truth is available)

In [ ]:
GROUND_TRUTH_PATH = None  # set to path if you have a ground-truth image

if GROUND_TRUTH_PATH:
    gt = cv2.imread(GROUND_TRUTH_PATH)
    psnr = compute_psnr(gt, result)
    print(f"PSNR: {psnr:.2f} dB")
else:
    print("No ground truth available — set GROUND_TRUTH_PATH to enable PSNR.")

## 8. Save the result

In [ ]:
from enhance.utils import save_result

out_path = IMAGE_PATH.replace('.jpg', '_dehazed.jpg').replace('.png', '_dehazed.png')
save_result(result, out_path)
print(f"Saved to: {out_path}")